In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv("twitter_training.csv",names=["id","about","target","comment"])
df.head()

In [ ]:
df["target"].value_counts()

In [ ]:
df.info()

In [ ]:
df=df[df["comment"].notna()]

In [ ]:
df.info()

In [ ]:
df=df[["target","comment"]]
df.sample(10)

In [ ]:
df=df.drop_duplicates()

In [ ]:
df=df[(df["target"]=="Positive") | (df["target"]=="Negative") | (df["target"]=="Neutral")]

In [ ]:
df["target"].value_counts()

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
tokenizer=Tokenizer(oov_token="Nothing")

In [ ]:
tokenizer.fit_on_texts(df["comment"])

In [ ]:
tokenizer.word_index

In [ ]:
tokenizer.word_counts

In [ ]:
tokenizer.document_count

In [ ]:
len(tokenizer.word_index)

In [ ]:
sequence=tokenizer.texts_to_sequences(df["comment"])
sequence

In [ ]:
tokenizer.word_index

In [ ]:
from keras.utils import pad_sequences


In [ ]:
sequences=pad_sequences(sequence,padding="pre")

In [ ]:
sequences

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
from tensorflow.keras.layers import Dense,SimpleRNN,Input,Embedding
from tensorflow.keras.models import Sequential

In [ ]:
label=LabelEncoder()
df["target"]=label.fit_transform(df["target"])
df["target"]

In [ ]:
X=sequences
y=df["target"].values
y

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=42,test_size=0.3)
X_test,X_valid,y_test,y_valid=train_test_split(X_test,y_test,random_state=42,test_size=0.5)

In [ ]:
X_train.shape[1]

In [ ]:
model=Sequential()
model.add(SimpleRNN(32,input_shape=(X_train.shape[1],1),return_sequences=False))
model.add(Dense(3,activation="softmax"))

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer="adam",metrics=["accuracy"],loss="sparse_categorical_crossentropy")

In [ ]:
model.fit(X_test,y_test,epochs=10,validation_data=(X_valid,y_valid),batch_size=64)

In [ ]:
model2=Sequential()
vocab_size=len(tokenizer.word_index)+1
model2.add(Input(shape=(X_train.shape[1],)))
model2.add(Embedding(input_dim=vocab_size,output_dim=64))
model2.add(SimpleRNN(32,return_sequences=True))
model2.add(SimpleRNN(32,return_sequences=False))
model2.add(Dense(3,activation="softmax"))
model2.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [ ]:
model2.compile(optimizer="adam",metrics=["accuracy"],loss="sparse_categorical_crossentropy")
model2.fit(X_test,y_test,epochs=100,validation_data=(X_valid,y_valid),batch_size=64,callbacks=[stopping_callback])

In [ ]:
import pickle
with open("tokenizer.pkl","wb") as file:
    pickle.dump(tokenizer,file)

In [ ]:
model2.save("Sentimentanalysis.h5")